In [0]:
!pip install py-ewr --upgrade
!pip uninstall -y kaleido
!pip install kaleido==0.2.1
!pip install numba
dbutils.library.restartPython()

# Define functions

In [0]:
from datetime import datetime
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
import os

from numba import njit

from shared.heatmap_variables import Variables
  
def read_in_flows(sites, variable_name, gauges):

  scenario_dfs = {
      short_scen_name: generate_variable_comparison_df(
          variable_name,
          sites,
          full_scen_name
      ).sort_index()
      for short_scen_name, full_scen_name in Variables.scenarios.items()
  }

  flows_by_gauge = {
      gauge: pd.concat(
          [df[[gauge]] for df in scenario_dfs.values()],
          axis=1
      ).set_axis(scenario_dfs.keys(), axis=1)
      for gauge in gauges
  }

  return flows_by_gauge

def get_heatmap(achievement_frequencies, xlim, flow_regimes, durations, spacing, scens, zrange):

    zmin, zmax = zrange
    base_scen, comp_scen = scens

    thresholds = np.arange(0, xlim+1)

    scen_lookup = {scen: i for i, scen in enumerate(Variables.scenarios.keys())}

    # get differential values

    base_s = scen_lookup[base_scen]
    comp_s = scen_lookup[comp_scen]

    diff = achievement_frequencies[comp_s] #- achievement_frequencies[base_s]

    diff = diff[:, 0, :].T

    diff = diff[:, ::spacing]
    thresholds = thresholds[::spacing]

    ## Generate plots
    ####
    if zmin < 0:
        color_continuous_scale = "RdBu"
    else:
        color_continuous_scale = "Blues"

    fig = px.imshow(
        diff,
        x=thresholds,
        y=durations,
        origin="lower",
        aspect="auto",
        color_continuous_scale=color_continuous_scale,
        zmin=zmin,
        zmax=zmax,
        labels={
            "x": "Threshold (ML/day)",
            "y": "Duration (days)",
        },
    )

    fig.add_trace(
        go.Contour(
            z=diff,
            x=thresholds,
            y=durations,
            contours=dict(
                start=5,
                end=zmax+5,
                size=5,
                coloring="none",
                showlabels=False,
            ),
            line=dict(
                color='black',
                width=1,
            ),
            showscale=False,
        ))

    fig.update_coloraxes(
        colorbar=dict(
            title=dict(
                text=f"% change from {base_scen} to {comp_scen}",
                side="right"
            )
        )
    )

    for x, label in flow_regimes:
        fig.add_vline(x=x, line_dash="dash", line_color="black", line_width=1)
        fig.add_annotation(x=x, y=durations.mean(), text=label, textangle=90, showarrow=False, xshift=12, yshift=-125, bgcolor="white", bordercolor="black", borderpad=2)

    ###
    # Format the plot
    fig.update_layout(
        height=600,
        width=1000,
        legend=dict(
            orientation="h",
            yanchor="top",
            y=-0.15,
            xanchor="center",
            x=0.5,
        ),
        margin=dict(t=20),
        plot_bgcolor="white",
        paper_bgcolor="white",
    )

    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor="black",
        mirror=True,

        ticks="outside",
        ticklen=6,
        tickwidth=1,
        tickcolor="black"        
    )

    fig.update_yaxes(
        showgrid=False,
        showline=True,
        linecolor="black",
        mirror=True,
        zeroline=True,
        zerolinecolor="black",
        zerolinewidth=1,
        
        ticks="outside",
        ticklen=6,
        tickwidth=1,
        tickcolor="black"
    )

    fig.update_xaxes(range=[0, xlim])

    return fig

import time

def generate_heatmap_fig(config, scen):

    achievement_frequencies = calc_achievement_frequencies(flows_df, config['xlim'], config['timing_windows'], config['durations'])

    abs_fig = get_heatmap(achievement_frequencies, config['xlim'], config['flow_regimes'], config['durations'], config['spacing'], ("scen", "scen"), (0, 100))  

    return abs_fig

def calc_achievement_frequencies(flow_df, xlim, timing_windows, duration_values):
    '''
    Parameters:
    flow_df (pd.DataFrame): Flow values for one given gauge, with columns for each scenario and datetime index for dates.
    xlim (int): maximum threshold value to consider. we will calculate thresholds of 0,1,2,3,...,xlim.
    timing_windows (list of tuples): each tuple is a pair of integers representing the start and end months of the timing_window.
    duration_values (np.array): array of durations.

    Returns:
    achievement_frequencies (np.ndarray): array of dimensions (num_scenarios, num_thresholds, num_timing_windows, num_durations)
                                          with each entry represents the frequency of achievement for the given EWR.
    '''
    
    # Preprocess dates
    months = flow_df.index.month.to_numpy()
    calendar_years = flow_df.index.year.to_numpy()

    wy = np.where(months >= 7, calendar_years + 1, calendar_years)
    year_lookup = np.unique(wy, return_inverse=True)[1] # water year indexed for each day in time series

    # Define thresholds and achievement_frequency shape
    thresholds = np.arange(0, xlim + 1)

    n_scenarios = flow_df.shape[1]
    achievement_frequencies = np.zeros((n_scenarios, len(thresholds), len(timing_windows), len(duration_values)), dtype=np.float32)

    # Convert flows to numpy
    flows = flow_df.to_numpy()

    ## Generate event cache
    for window_index, (sm, em) in enumerate(timing_windows):
        if sm <= em:
            timing_mask = ((months >= sm) & (months <= em))
        else:
            timing_mask = ((months >= sm) | (months <= em))

        # Filter flows for timing window
        flows_in_timing_window = np.where(timing_mask[:, None], flows, -np.inf) # replace flow values outside timing window with -inf
        timing_window_length = int(np.ceil(timing_mask.sum() / (year_lookup.max() + 1)))

        import time

        start = time.perf_counter()

        calc_frequencies(flows_in_timing_window, year_lookup, thresholds, duration_values, achievement_frequencies, window_index, timing_window_length)

        end = time.perf_counter()
        print(f"Finished in {end-start} seconds")

    return achievement_frequencies

@njit
def get_year_boundaries(year_lookup, n_years):
    year_starts = np.empty(n_years, dtype=np.int32)
    year_ends = np.empty(n_years, dtype=np.int32)

    year_starts[:] = -1
    year_ends[:] = -1

    for day_index in range(len(year_lookup)):
        year = year_lookup[day_index]

        if year_starts[year] == -1:
            year_starts[year] = day_index

        year_ends[year] = day_index + 1

    return year_starts, year_ends

@njit
def calc_annual_max_threshold_nonconsecutive(
    flow_values,
    start_index,
    end_index,
    scen_index,
    duration,
):
    n_days = end_index - start_index

    if duration > n_days:
        return -np.inf

    year_flows = np.sort(
        flow_values[start_index:end_index, scen_index]
    )

    return year_flows[-duration]

@njit
def calc_annual_max_threshold(flow_values, start_index, end_index, scen_index, duration, version='default'):
    '''
    For every consecutive window of length `duration`, find the minimum threshold in that window,
    and then find the largest of those minimum thresholds. i.e. what is the maximum threshold in this year that the flows exceed for this given duration.

    values: np.array of flow values. This is filtered for a specific water year, scenario, and window already.
    duration: int
    '''
    
    if duration > (end_index - start_index):
        return -np.inf

    annual_max_threshold = -np.inf


    if version == 'default':
        for window_start in range(start_index, end_index - duration + 1):
            window_min_thresh = np.min(flow_values[window_start:window_start + duration, scen_index] )

            if window_min_thresh > annual_max_threshold:
                annual_max_threshold = window_min_thresh

        return annual_max_threshold

    else: # non-consecutive 
      for window_start in range(start_index, end_index - duration + 1):
          window_min_thresh = np.min(flow_values[window_start:window_start + duration, scen_index] )

          if window_min_thresh > annual_max_threshold:
              annual_max_threshold = window_min_thresh

      return annual_max_threshold


@njit
def calc_frequencies(flows, year_lookup, thresholds, duration_values, achievement_frequencies, window_index, timing_window_length):

    n_days, n_scenarios = flows.shape
    n_years = year_lookup.max() + 1
    n_durations = len(duration_values)

    # Locate the start and end of each water year.
    year_starts, year_ends = get_year_boundaries(year_lookup, n_years)

    # Calculate the maximum achieved thresholds for each scenario, duration, year. 
    # i.e. for each individual year what is the maximum threshold flows exceed for a given duration
    annual_maximum_thresholds = np.empty((n_scenarios, n_durations, n_years), dtype=flows.dtype)
    
    for scen_index in range(n_scenarios):  
        for year_index, (year_start, year_end) in enumerate(zip(year_starts, year_ends)):
            for dur_index in range(n_durations):
                duration = duration_values[dur_index]

                if duration > timing_window_length:
                    annual_maximum_thresholds[scen_index, dur_index, year_index] = -np.inf
                else:
                    annual_maximum_thresholds[scen_index, dur_index, year_index] = calc_annual_max_threshold_nonconsecutive(flows, year_start, year_end, scen_index, duration)

    # Convert annual maximum thresholds to achievement frequencies.
    for dur_index in range(n_durations):
        for scen_index in range(n_scenarios):
            sorted_annual_max_thresholds = np.sort(annual_maximum_thresholds[scen_index, dur_index, :])
            n_unsuccessful_years = np.searchsorted(sorted_annual_max_thresholds, thresholds) # for each threshold, get the number of years which DONT achieve this for the required duration
            achievement_frequencies[scen_index, :, window_index, dur_index] = (n_years - n_unsuccessful_years) * 100.0 / n_years

            # note, it is necessary to get the years which are less than the threshold, and then subtract this from the total number of years, because the searchsorted function
            # is only compatible with 'numba' under 'left' searching


# demo

In [0]:
config = {
    "timing_windows": [(7, 6)],
    "xlim": 60000,
    "flow_regimes":  [],#[(800, "Base flow"), (4000, "Small fresh"), (12000, "Large fresh"), (15500, "Wetland large fresh"), (28000, "Small overbank"), (40000, "Large overbank")],
    "durations": np.arange(1, 101),
    "spacing": 100,
}

abs_fig = generate_heatmap_fig(config)

abs_fig.show()
